# Model Monitoring: Data Drift Detection and Alerting

Monitors the deployed CarePath readmission endpoint for data drift by comparing live requests (captured via Data Capture) against a baseline built from the training data.

## 1. Setup
Install SageMaker SDK v2, import libraries, and connect to the deployed endpoint.

In [1]:
%pip install -q "sagemaker>=2.200,<3"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json, time
import boto3, joblib
import numpy as np
import pandas as pd
import sagemaker
from sagemaker import get_execution_role
from sagemaker.model_monitor import DefaultModelMonitor, CronExpressionGenerator
from sagemaker.model_monitor.dataset_format import DatasetFormat

session = sagemaker.Session()
bucket = session.default_bucket()
role = get_execution_role()
sm = boto3.client("sagemaker")

endpoint_name = "carepath-readmission-endpoint-1790992239"
PROJECT_PREFIX = "carepath-analytics"
baseline_prefix = f"{PROJECT_PREFIX}/monitoring/baseline"
reports_prefix = f"{PROJECT_PREFIX}/monitoring/reports"

print(sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"])

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


InService


## 2. Create baseline dataset
The baseline uses the same patient-level training split and preprocessor as `train-model.ipynb`, so it matches exactly what the endpoint receives: transformed features, CSV, no header, no label.

In [3]:
from sklearn.model_selection import train_test_split

artifacts = "../training/artifacts"
df = pd.read_csv(f"{artifacts}/diabetes_processed.csv")
preprocessor = joblib.load(f"{artifacts}/preprocessor.joblib")

categorical_cols = ["age", "admission_type", "discharge_disposition", "admission_source",
                    "diag_1_group", "diag_2_group", "diag_3_group",
                    "max_glu_serum", "A1Cresult", "change", "diabetesMed"]
numeric_cols = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
                "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]
feature_cols = categorical_cols + numeric_cols

# Recreate the same patient-level split (random_state=42)
patients = df.groupby(df["patient_nbr"].astype(str))["readmitted_30d"].max().reset_index()
train_p, temp_p = train_test_split(patients, test_size=0.30, random_state=42, stratify=patients["readmitted_30d"])
val_p, test_p = train_test_split(temp_p, test_size=0.50, random_state=42, stratify=temp_p["readmitted_30d"])

pid = df["patient_nbr"].astype(str)
X_train = df.loc[pid.isin(set(train_p["patient_nbr"])), feature_cols]
X_test = df.loc[pid.isin(set(test_p["patient_nbr"])), feature_cols]

X_train_t = preprocessor.transform(X_train)
X_train_t = X_train_t.toarray() if hasattr(X_train_t, "toarray") else X_train_t

baseline_file = "baseline.csv"
pd.DataFrame(X_train_t).to_csv(baseline_file, header=False, index=False)
baseline_data_uri = session.upload_data(baseline_file, bucket=bucket, key_prefix=f"{baseline_prefix}/data")

print("Baseline shape:", X_train_t.shape)
print("Baseline data:", baseline_data_uri)

Baseline shape: (71178, 104)
Baseline data: s3://sagemaker-us-east-1-135844594927/carepath-analytics/monitoring/baseline/data/baseline.csv


## 3. Generate baseline statistics and constraints
Model Monitor profiles the baseline data and suggests constraints (data types, completeness, value ranges). Live traffic is later checked against these.

In [4]:
monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
    sagemaker_session=session,
)

monitor.suggest_baseline(
    baseline_dataset=baseline_data_uri,
    dataset_format=DatasetFormat.csv(header=False),
    output_s3_uri=f"s3://{bucket}/{baseline_prefix}/results",
    wait=True,
    logs=False,
)

constraints = pd.json_normalize(monitor.baseline_statistics().body_dict["features"])
print("Features profiled:", len(constraints))
constraints[["name", "inferred_type", "numerical_statistics.mean"]].head()

/opt/conda/lib/python3.12/site-packages/sagemaker/model_monitor/model_monitoring.py:176: SageMakerV2DeprecationWarning: DefaultModelMonitor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.shapes.MonitoringSchedule`.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
DefaultModelMonitor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.shapes.MonitoringSchedule`.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
/opt/conda/lib/python3.12/site-packages/sagemaker/processing.py:138: SageMakerV2DeprecationWarning: Processor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `DataProcessor` (`from sagemaker.mlops.processing

...............................................................................................!Features profiled: 104


,name,inferred_type,numerical_statistics.mean
0,_c0,Fractional,0.001756
1,_c1,Fractional,0.006744
2,_c2,Fractional,0.016466
3,_c3,Fractional,0.038017
4,_c4,Fractional,0.095057


## 4. Model Monitor availability change
SageMaker Model Monitor scheduled jobs are no longer available to new AWS customers (effective 7/30/26), so `create_monitoring_schedule` fails in this account. Following AWS's recommended replacement pattern, this notebook:
- Keeps Data Capture on the endpoint and the Model Monitor baseline generated in Section 3
- Runs a custom data drift check in Python against the captured traffic
- Publishes drift metrics to Amazon CloudWatch for alerting

## 5. Endpoint traffic
Before sending traffic for monitoring, endpoint predictions are validated against the local model to catch training-serving skew.

### 5a. Training-serving skew check
Initial endpoint predictions were near zero (mean 0.0008). Root cause: the model was trained on a **sparse** matrix, where XGBoost treats empty entries as **missing**, but the endpoint received **dense** CSV where those entries are explicit **zeros**, sending rows down different tree paths.

- Local sparse prediction: mean 0.3996
- Local dense prediction: mean 0.0008 (matches endpoint)

Fix: serialize zeros as blank CSV fields so the endpoint treats them as missing, matching training. Recommended long-term fix: retrain with dense features (`sparse_output=False`) so training and serving use the same format.

In [8]:
import xgboost as xgb

booster = xgb.Booster()
booster.load_model(f"{artifacts}/xgboost-model")

X_sparse = preprocessor.transform(X_test.iloc[:200])
p_sparse = booster.predict(xgb.DMatrix(X_sparse))
p_dense = booster.predict(xgb.DMatrix(X_sparse.toarray()))

print("Sparse (how it was trained):  mean", p_sparse.mean().round(4), "| max", p_sparse.max().round(4))
print("Dense (what endpoint gets):   mean", p_dense.mean().round(4), "| max", p_dense.max().round(4))

Sparse (how it was trained):  mean 0.3996 | max 0.7762
Dense (what endpoint gets):   mean 0.0008 | max 0.006


In [9]:
runtime = boto3.client("sagemaker-runtime")

X_test_t = preprocessor.transform(X_test)
X_test_t = X_test_t.toarray() if hasattr(X_test_t, "toarray") else X_test_t

# Serialize zeros as blanks so the endpoint treats them as missing, matching sparse training
def to_csv_payload(row):
    return ",".join("" if v == 0 else str(v) for v in row)

for i in range(5):
    resp = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="text/csv",
        Body=to_csv_payload(X_test_t[i]),
    )
    print(round(float(resp["Body"].read().decode()), 4), "| local sparse:", round(float(p_sparse[i]), 4))

0.0871 | local sparse: 0.0871
0.2801 | local sparse: 0.2801
0.535 | local sparse: 0.535
0.2023 | local sparse: 0.2023
0.3456 | local sparse: 0.3456


### 5b. Send test traffic to the endpoint
Sends 200 test-set patients to the endpoint using the corrected payload format. Data Capture logs each request and response to S3 for the drift check.

In [10]:
n_requests = 200
predictions = []

for row in X_test_t[:n_requests]:
    payload = to_csv_payload(row)
    resp = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="text/csv",
        Body=payload,
    )
    predictions.append(float(resp["Body"].read().decode()))

print("Requests sent:", len(predictions))
print("Sample predicted probabilities:", np.round(predictions[:5], 4))

Requests sent: 200
Sample predicted probabilities: [0.0871 0.2801 0.535  0.2023 0.3456]


In [11]:
preds = np.array(predictions)
print("Mean:", preds.mean().round(4), "| Max:", preds.max().round(4))
print("Above 0.5:", (preds > 0.5).sum(), "of", len(preds))

Mean: 0.3996 | Max: 0.7762
Above 0.5: 42 of 200


## 6. Load captured endpoint traffic
Data Capture writes each request and response as JSON Lines to S3. This parses the captured requests back into a feature matrix for comparison against the baseline.

In [12]:
s3 = boto3.client("s3")
capture_prefix = f"{PROJECT_PREFIX}/datacapture/{endpoint_name}"

keys = []
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=capture_prefix):
    keys += [obj["Key"] for obj in page.get("Contents", [])]

rows = []
for key in keys:
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read().decode()
    for line in body.strip().splitlines():
        record = json.loads(line)
        csv_input = record["captureData"]["endpointInput"]["data"]
        rows.append([float(v) if v else 0.0 for v in csv_input.split(",")])

captured_df = pd.DataFrame(rows)
print("Capture files:", len(keys))
print("Captured requests:", captured_df.shape)

Capture files: 2
Captured requests: (205, 104)


## 7. Custom data drift check
Compares captured traffic to the training baseline, feature by feature:
- **Numeric features:** Kolmogorov-Smirnov test (p < 0.01 flags drift)
- **One-hot categorical features:** absolute change in proportion (> 0.10 flags drift)

The drift score is the share of features flagged.

In [13]:
from scipy.stats import ks_2samp

with open(f"{artifacts}/transformed_feature_names.json") as f:
    feature_names = json.load(f)

def drift_report(baseline, live, names, p_thresh=0.01, prop_thresh=0.10):
    results = []
    for i, name in enumerate(names):
        b, l = baseline[:, i], live[:, i]
        if name.startswith("numeric__"):
            stat, p = ks_2samp(b, l)
            drifted = p < p_thresh
            results.append({"feature": name, "test": "KS", "value": round(stat, 4), "drift": drifted})
        else:
            diff = abs(b.mean() - l.mean())
            results.append({"feature": name, "test": "proportion", "value": round(diff, 4), "drift": diff > prop_thresh})
    report = pd.DataFrame(results)
    score = report["drift"].mean()
    return report, score

report, drift_score = drift_report(X_train_t, captured_df.values, feature_names)

print(f"Drift score: {drift_score:.2%} of features drifted")
report[report["drift"]]

Drift score: 8.65% of features drifted


,feature,test,value,drift
11,categorical__admission_type_Emergency,proportion,0.2627,True
14,categorical__admission_type_Unknown,proportion,0.3615,True
17,categorical__discharge_disposition_Discharged ...,proportion,0.1641,True
39,categorical__discharge_disposition_Unknown,proportion,0.3688,True
42,categorical__admission_source_Emergency Room,proportion,0.1004,True
97,numeric__num_lab_procedures,KS,0.1555,True
100,numeric__number_outpatient,KS,0.1357,True
102,numeric__number_inpatient,KS,0.1495,True
103,numeric__number_diagnoses,KS,0.1980,True


In [14]:
rng = np.random.default_rng(42)
random_idx = rng.choice(len(X_test_t), size=len(captured_df), replace=False)

random_report, random_score = drift_report(X_train_t, X_test_t[random_idx], feature_names)

print(f"Captured traffic (first 200 rows): {drift_score:.2%} drifted")
print(f"Random test sample:                {random_score:.2%} drifted")

Captured traffic (first 200 rows): 8.65% drifted
Random test sample:                0.00% drifted


### Drift check results
| Traffic sent | Drift score |
|---|---|
| First 200 test patients (earliest records) | 8.65% |
| Random sample of test patients (same size) | 0.00% |

The first rows of the dataset come from the earliest encounters, which differ in coding (e.g., more "Unknown" admission types) and utilization patterns (lab procedures, prior visits, diagnoses count). The check flags this non-random slice while showing no drift on a random sample, which confirms it detects real distribution shift without false alarms.

## 8. Publish drift metrics and alarm to CloudWatch
The drift score and drifted-feature count are published as custom CloudWatch metrics. An alarm fires when the drift score exceeds 5%, replacing Model Monitor's built-in alerting. In production, this check would run on a schedule via Amazon EventBridge.

In [15]:
cw = boto3.client("cloudwatch")
namespace = "CarePath/ModelMonitoring"
dimensions = [{"Name": "EndpointName", "Value": endpoint_name}]

cw.put_metric_data(
    Namespace=namespace,
    MetricData=[
        {"MetricName": "DataDriftScore", "Dimensions": dimensions,
         "Value": float(drift_score), "Unit": "None"},
        {"MetricName": "DriftedFeatureCount", "Dimensions": dimensions,
         "Value": int(report["drift"].sum()), "Unit": "Count"},
    ],
)

alarm_name = f"carepath-data-drift-{endpoint_name}"
cw.put_metric_alarm(
    AlarmName=alarm_name,
    AlarmDescription="Fires when more than 5% of features drift from the training baseline",
    Namespace=namespace,
    MetricName="DataDriftScore",
    Dimensions=dimensions,
    Statistic="Maximum",
    Period=300,
    EvaluationPeriods=1,
    Threshold=0.05,
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
)

print("Published drift score:", round(drift_score, 4))
print("Alarm created:", alarm_name)

Published drift score: 0.0865
Alarm created: carepath-data-drift-carepath-readmission-endpoint-1790992239


## 9. Verify alarm state
Confirms the CloudWatch alarm moved to ALARM after the drifted traffic was published.

In [16]:
state = cw.describe_alarms(AlarmNames=[alarm_name])["MetricAlarms"][0]
print("State:", state["StateValue"])
print("Reason:", state["StateReason"])

State: OK
Reason: Threshold Crossed: no datapoints were received for 1 period and 1 missing datapoint was treated as [NonBreaching].


The first check ran before the metric datapoint was ingested, so the alarm evaluated an empty period as OK. Publishing the score over several minutes lets CloudWatch evaluate it:

In [17]:
for i in range(4):
    cw.put_metric_data(
        Namespace=namespace,
        MetricData=[{"MetricName": "DataDriftScore", "Dimensions": dimensions,
                     "Value": float(drift_score), "Unit": "None"}],
    )
    print(f"Published {i + 1}/4")
    time.sleep(60)

state = cw.describe_alarms(AlarmNames=[alarm_name])["MetricAlarms"][0]
print("State:", state["StateValue"])
print("Reason:", state["StateReason"])

Published 1/4
Published 2/4
Published 3/4
Published 4/4
State: ALARM
Reason: Threshold Crossed: 1 datapoint [0.08653846153846154 (03/10/26 02:18:00)] was greater than the threshold (0.05).


## 10. Clean up
Deletes the endpoint, endpoint configuration, and alarm to stop charges. Captured data, baseline results, and metrics remain in S3 and CloudWatch for reporting.

In [18]:
config_name = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointConfigName"]

sm.delete_endpoint(EndpointName=endpoint_name)
sm.delete_endpoint_config(EndpointConfigName=config_name)
cw.delete_alarms(AlarmNames=[alarm_name])

print("Deleted endpoint:", endpoint_name)
print("Remaining endpoints:", [e["EndpointName"] for e in sm.list_endpoints()["Endpoints"]])

Deleted endpoint: carepath-readmission-endpoint-1790992239
Remaining endpoints: ['carepath-readmission-endpoint-1790992239']


Endpoint deletion is asynchronous, so it briefly still appears while in `Deleting` status. Confirming it is fully removed:

In [19]:
print([(e["EndpointName"], e["EndpointStatus"]) for e in sm.list_endpoints()["Endpoints"]])

[]
